#### Task 1 Creating the data frames from csv files

In [ ]:
import os
import pandas as pd

#The folder directory where the source .csv files are located
dir = r'your directory here'

#This function handles the entire process from .csv to dataframe in several steps:

def df_creation(directory):
    # Step 1: Obtains the contents from 'dir' variable. 
    contents = os.listdir(dir)

    if not contents:
        print('The directory is empty')
        return
    
    # Step 2: Checks which files in 'dir' are .csv and records them to csv_files list.
    csv_files = []
    for i in contents:
        if i.endswith('.csv'):
            csv_files.append(i)

    if not csv_files:
        print('No csv files found')
        return
    
    # Step 3: Loops through csv_files list, strips them from their .csv extentions and saving the results to 'df_name'
    #'globals()[df_name]' creates a new variable with the name from 'df_name', reads the .csv files and drops any duplicated rows to ensure data integrity
    #records the created dataframes names to 'df_names' as it would be easier to check the list of available dataframes later in the task.
    df_names = []
    try:
        for file in csv_files:
            df_name = file.replace('.csv','')
            globals()[df_name] = pd.read_csv(dir + '/' + file, sep = ';').drop_duplicates()
            df_names.append(df_name)
    except Exception as e:
        print(f'Error loading {file}: {e}')
    
    return df_names

dfs_list = df_creation(dir)

In [3]:
#Count of each table
df_count = {}

for df_name in dfs_list:
    df_count[df_name] = len(globals()[df_name])

df_count

{'product': 504,
 'product_category': 4,
 'product_subcategory': 37,
 'sales': 31465,
 'sales_details': 121317,
 'special_offer': 16}

####  Quantity sold per product

In [ ]:
#Creating 'orders_products' table
orders_products = pd.merge(sales_details, product[['ProductID','Name']], how='left', on='ProductID')
orders_products.head(5)

,SalesOrderID,SalesOrderDetailID,CarrierTrackingNumber,OrderQty,ProductID,SpecialOfferID,UnitPrice,UnitPriceDiscount,LineTotal,rowguid,ModifiedDate,Name
0,43659,1,4911-403C-98,1,776,1,2024.994,0.0,2024.994,{B207C96D-D9E6-402B-8470-2CC176C42283},2005-07-01 00:00:00,"Mountain-100 Black, 42"
1,43659,2,4911-403C-98,3,777,1,2024.994,0.0,6074.982,{7ABB600D-1E77-41BE-9FE5-B9142CFC08FA},2005-07-01 00:00:00,"Mountain-100 Black, 44"
2,43659,3,4911-403C-98,1,778,1,2024.994,0.0,2024.994,{475CF8C6-49F6-486E-B0AD-AFC6A50CDD2F},2005-07-01 00:00:00,"Mountain-100 Black, 48"
3,43659,4,4911-403C-98,1,771,1,2039.994,0.0,2039.994,{04C4DE91-5815-45D6-8670-F462719FBCE3},2005-07-01 00:00:00,"Mountain-100 Silver, 38"
4,43659,5,4911-403C-98,1,772,1,2039.994,0.0,2039.994,{5A74C7D2-E641-438E-A7AC-37BF23280301},2005-07-01 00:00:00,"Mountain-100 Silver, 42"


In [ ]:
#Creating 'product_qty' table
product_qty = pd.pivot_table(orders_products, index='Name', values= 'OrderQty', aggfunc='sum').sort_values('OrderQty', ascending=False)
product_qty.index.name = 'Product Name'

In [ ]:
#Total quantity sold
print('Total quantity sold:', product_qty['OrderQty'].sum())

Total quantity sold: 274914


In [ ]:
#Top 5 most ordered products
product_qty.head(5)

,OrderQty
Product Name,
AWC Logo Cap,8311
Water Bottle - 30 oz.,6815
"Sport-100 Helmet, Blue",6743
"Long-Sleeve Logo Jersey, L",6592
"Sport-100 Helmet, Black",6532


In [ ]:
#Top 5 least ordered products
product_qty.tail(5)

,OrderQty
Product Name,
"LL Touring Frame - Blue, 62",15
LL Road Seat/Saddle,10
"LL Mountain Frame - Black, 40",8
"ML Mountain Frame-W - Silver, 38",7
"LL Touring Frame - Blue, 58",4


### Number of orders on sale

In [ ]:
#Joining sales table with sales_details to obrain ProductID and SpecialOfferID
orders_offerid = pd.merge(sales, sales_details[['SalesOrderID','SpecialOfferID','ProductID']], how='left', on='SalesOrderID')

#Joining order_offerid with product table to obrain the names of the products.
order_productname = pd.merge(orders_offerid,product[['ProductID','Name']], how='left', on='ProductID')

#Checking which is the special offer ID with no discount, as it does not qualify as a discount due to it's 0% discount value.
not_disc = special_offer[special_offer['DiscountPct'] == 0]['SpecialOfferID'].iloc[0]

#Filtering out SpecialOfferID that is not a sale 
order_productname = order_productname[order_productname['SpecialOfferID'] != not_disc]

#Pivoting 'order_productname' to get the highest bought product which was on sale. 
order_nosales = pd.pivot_table(order_productname, index='Name', values= 'SalesOrderID', aggfunc='count').sort_values('SalesOrderID', ascending=False)
order_nosales.columns = ['Order Count']
order_nosales.index.name = 'Product Name'


In [ ]:
#Product with highest number of sales while discounte
order_nosales.head(1)

,Order Count
Product Name,
Patch Kit/8 Patches,844


#### Adding "unit_discount" and "total_discount" columns to sales_details table

In [11]:
#Calculating the discount amount per unit
sales_details['unit_discount'] = sales_details['UnitPrice'] * sales_details['UnitPriceDiscount']

#Calculating the total discount per product in an order
sales_details['total_discount']= sales_details['OrderQty'] * sales_details['unit_discount']

#Calculating the price per unit after discount is applied
sales_details['discounted_price'] = sales_details['UnitPrice'] - sales_details['unit_discount']

 #### Customers and orders placed

In [ ]:
#Pivoting 'sales' table for 'CustomerID' and 'SalesOrderID' with 'count' as aggregation, sorting in descending order.
customer_orders = pd.pivot_table(sales, index= 'CustomerID', values='SalesOrderID', aggfunc='count').sort_values('SalesOrderID', ascending=False)
customer_orders.columns = ['Total Orders']

#Taking the maxmum value of the 'customer_orders' table and filtering the table to get the customer(s) with highest orders placed.
customer_orders[customer_orders['Total Orders'] == customer_orders['Total Orders'].max()]

,Total Orders
CustomerID,
11091,28
11176,28


#### Create frequency classification and monetary_value

In [ ]:
#Creating "frequency" column

customer_orders.reset_index(inplace= True)

for i, cust in enumerate(customer_orders['Total Orders']): 
    if cust == 1:
        customer_orders.loc[i, 'frequency'] = 'New'
    elif cust > 1 and cust < 4:
        customer_orders.loc[i, 'frequency'] = 'Repeated'
    else:
        customer_orders.loc[i, 'frequency'] = 'Fan'

#Creating "monetary_value" column

customer_purchase = pd.pivot_table(sales, index='CustomerID', values='SubTotal', aggfunc='sum')
customer_purchase.reset_index(inplace= True)

for i,cust in enumerate(customer_purchase['SubTotal']):
    if cust < 100:
        customer_purchase.loc[i, 'monetary_value'] = 'Frugal Spender'
    elif cust > 100 and cust < 10000:
        customer_purchase.loc[i, 'monetary_value'] = 'Medium Spender'
    else:
        customer_purchase.loc[i, 'monetary_value'] = 'High Spender'

#Merging 'customer_purchase' and 'customer_orders' tables into 'customer_classification' table as a final result.
customer_classification = pd.merge(customer_orders, customer_purchase[['CustomerID','SubTotal','monetary_value']], how= 'left', on= 'CustomerID')

In [14]:
pd.crosstab(customer_classification['frequency'], customer_classification['monetary_value'], margins=True)

monetary_value,Frugal Spender,High Spender,Medium Spender,All
frequency,,,,
Fan,3,473,317,793
New,6832,0,4817,11649
Repeated,848,4,5825,6677
All,7683,477,10959,19119


In [15]:
#The best customers are the Medium spenders with "repeated" frequency. These customers have multiple order, they are not many, but the value of the orders is high.
#Their large transactions are better for the business and it costs less operationally to fulfill their orders. 

#The worst cusomters are "Fan" and "Frugal Spender" as they place a lot of orders but with barely any value. The company spends a lot of resources on their orders 
#however the return is minimal. 

#### Table of product categories, total sales, total discount, and total orders per category.

In [ ]:
#Merging 'sales_details' with 'product' to obrain 'ProductSubcategoryID'
df = pd.merge(sales_details[['ProductID','SalesOrderID','LineTotal','total_discount']], product[['ProductID', 'ProductSubcategoryID']], how='left', on='ProductID')

#Merging 'df' with 'product_subcategory' to obtain 'ProductCategoryID'
df1 = pd.merge(df, product_subcategory[['ProductSubcategoryID','ProductCategoryID']], how='left', on='ProductSubcategoryID' )

#Merging 'df1' with 'product_category' to obtain the 'Name' of the product
df_final = pd.merge(df1, product_category[['ProductCategoryID', 'Name']], how='left', on='ProductCategoryID')

#Adding '1' to each row in order to count the number of orders with aggfunc='sum' when pivoting the table next. 
#Each row represents one order per product, therefore this logic is valid.
df_final['order_count'] = 1

#Pivoting the merged tables to obtain final result.
product_summary =pd.pivot_table(df_final, index='Name', values=['LineTotal','total_discount','order_count'], aggfunc='sum').astype(int)
product_summary


,LineTotal,order_count,total_discount
Name,,,
Accessories,1272072,41194,6688
Bikes,94651172,40031,494640
Clothing,2120542,21394,20964
Components,11802593,18698,5214


In [17]:
#Finding the procut with highest sales
product_summary[product_summary['LineTotal'] == product_summary['LineTotal'].max()]

,LineTotal,order_count,total_discount
Name,,,
Bikes,94651172,40031,494640


In [18]:
#Finding the product with most orders
product_summary[product_summary['order_count'] == product_summary['order_count'].max()]

,LineTotal,order_count,total_discount
Name,,,
Accessories,1272072,41194,6688


#### KPIs

##### KPI #1 - Product margin per sub-category

In [ ]:
#Merging only products with subcategory available
df = pd.merge(product, product_subcategory[['ProductSubcategoryID','ProductCategoryID']], how='inner', on='ProductSubcategoryID')
df1 = pd.merge(df, product_category[['ProductCategoryID', 'Name']], how='left', on='ProductCategoryID')
df1.rename(columns={'Name_x':'Product Name', 'Name_y':'Category'}, inplace=True)

#Calculating average margin
df1['AverageMargin'] = df['ListPrice'] - df1['StandardCost']

#Pivoting df1 table  to create 'margin' table
margin = pd.pivot_table(df1, index='Category',values='AverageMargin', aggfunc='mean' )
margin.reset_index(inplace=True)

#Pivoting df1 table  to create 'manufacture' table
manufacture = pd.pivot_table(df1, index='Category',values='DaysToManufacture', aggfunc='mean' )
manufacture.reset_index(inplace=True)

#Mering 'margin' and 'manufacture' tables to obtain final KPI table. 
margin_manufact_kpi = pd.merge(margin, manufacture[['Category','DaysToManufacture']], how='left', on='Category').round(1)
margin_manufact_kpi

,Category,AverageMargin,DaysToManufacture
0,Accessories,21.1,0.0
1,Bikes,637.3,4.0
2,Clothing,26.2,0.0
3,Components,201.7,1.1


##### KPI#2 - Most successfull sales campaigns

In [ ]:
#Mergind 'sales_details' and 'special_offer' tables to obrain product names and filtering DiscountPct so the table includes only promotional campaigns.
sales_sale = pd.merge(sales_details, special_offer[['SpecialOfferID','Description','DiscountPct']], how='left', on='SpecialOfferID')
sales_sale = sales_sale[sales_sale['DiscountPct'] > 0 ]

#Pivoting the 'sales_sale' table
discount_sales = pd.pivot_table(sales_sale, index='Description', values='LineTotal', aggfunc='sum').astype(int).sort_values('LineTotal', ascending=False)
discount_sales.reset_index(inplace=True)
discount_sales.rename(columns={'Description':'Sale Type', 'LineTotal':'Total Sales'}, inplace=True)
discount_sales

,Sale Type,Total Sales
0,Volume Discount 11 to 14,4896451
1,Volume Discount 15 to 24,1037643
2,Touring-1000 Promotion,612324
3,Touring-3000 Promotion,458091
4,Mountain-100 Clearance Sale,250927
5,Volume Discount 25 to 40,124148
6,Road-650 Overstock,49986
7,Mountain-500 Silver Clearance Sale,25899
8,Sport Helmet Discount-2003,9100
9,Sport Helmet Discount-2002,7448


##### KPI#3 - Saler per year and month

In [ ]:
import datetime as dt

#Converting 'OrderDate' column to datatime format and extracting two new columns 'OrderMonth' and 'OrderYear'
sales['OrderDate'] = pd.to_datetime(sales['OrderDate'])
sales['OrderMonth'] = sales['OrderDate'].dt.month
sales['OrderYear'] = sales['OrderDate'].dt.year

#Using .crosstab() to display SubTotal by Year and Month. 
period_sales = pd.crosstab(sales['OrderYear'], sales['OrderMonth'], values=sales['SubTotal'], aggfunc='sum',margins=True).fillna(0).astype(int)
period_sales.rename(columns= {'All':'Total'}, inplace=True)
period_sales.rename(index= {'All':'Total'}, inplace=True)
period_sales.reset_index(inplace=True)
period_sales

OrderMonth,OrderYear,1,2,3,4,5,6,7,8,9,10,11,12,Total
0,2005,0,0,0,0,0,0,962716,2044600,1639840,1358050,2868129,2458472,11331808
1,2006,1309863,2451605,2099415,1546592,2942672,1678567,2894054,4147192,3235826,2217544,3388911,2762527,30674773
2,2007,1756407,2873936,2049529,2371677,3443525,2542671,3554092,5068341,5059473,3364506,4683867,5243008,42011037
3,2008,3009197,4167855,4221323,3820583,5194121,5364840,50840,0,0,0,0,0,25828762
4,Total,6075467,9493397,8370268,7738853,11580319,9586079,7461704,11260133,9935139,6940101,10940907,10464007,109846381


In [ ]:
#Creating an .XLSX file 'Insights' including the 3 data frames with KPIs
with pd.ExcelWriter(r'C:\Users\adisq\Desktop\Desktop\INDEAVR\Insights.xlsx', engine='openpyxl') as writer:
    margin_manufact_kpi.to_excel(writer, sheet_name='Margins', index=False)
    discount_sales.to_excel(writer, sheet_name='Discount Sales', index=False)
    period_sales.to_excel(writer, sheet_name='Period Sales', index=False)